# Sprint 1 — reproducible transaction EDA

This notebook is a thin, reviewable presentation layer over the bounded-memory reporting scripts. It profiles the canonical feature store when available and otherwise falls back to the governed AMLworld raw CSV. It does not load the full transaction table into notebook memory.

**Interpretation boundary:** AMLworld is synthetic. Its label prevalence and generated behavior are not estimates of production AML prevalence or deployment performance.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'config' / 'datasets.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config' / 'datasets.json').exists(), 'Run from the repository or notebooks directory'
ROOT

## Select the measured dataset

A nonempty processed feature store takes precedence. Set `MAX_ROWS` to a positive integer for a wiring check; leave it as `None` for the full Sprint 1 report.

In [ ]:
FEATURE_ROOT = ROOT / 'data' / 'processed' / 'feature_store'
RAW_CSV = ROOT / 'data' / 'raw' / 'ibm_aml_hi_small' / 'HI-Small_Trans.csv'
INPUT = FEATURE_ROOT if FEATURE_ROOT.exists() and any(FEATURE_ROOT.rglob('*.parquet')) else RAW_CSV
REPORT = ROOT / 'reports' / 'eda'
DICTIONARY = ROOT / 'docs' / 'data_dictionary.csv'
MAX_ROWS = None  # Example smoke test: 100_000
{'input': str(INPUT), 'report': str(REPORT), 'max_rows': MAX_ROWS}

In [ ]:
eda_command = [
    sys.executable, str(ROOT / 'scripts' / 'run_eda.py'),
    '--input', str(INPUT), '--output-dir', str(REPORT),
]
dictionary_command = [
    sys.executable, str(ROOT / 'scripts' / 'build_data_dictionary.py'),
    '--input', str(INPUT), '--output', str(DICTIONARY),
]
if MAX_ROWS is not None:
    eda_command += ['--max-rows', str(MAX_ROWS)]
    dictionary_command += ['--max-rows', str(MAX_ROWS)]
subprocess.run(eda_command, cwd=ROOT, check=True)
subprocess.run(dictionary_command, cwd=ROOT, check=True)

## Scope, provenance, and headline diagnostics

Always check `is_bounded_sample` before interpreting results. Input-order regressions in the raw CSV justify chronological sorting in the feature pipeline; they are not model results.

In [ ]:
summary = json.loads((REPORT / 'summary.json').read_text(encoding='utf-8'))
pd.Series({
    'rows_profiled': summary['profile_scope']['rows'],
    'bounded_sample': summary['profile_scope']['is_bounded_sample'],
    'minimum_utc': summary['time']['minimum_utc'],
    'maximum_utc': summary['time']['maximum_utc'],
    'positive_rate': summary['class_balance']['positive_rate'],
    'timestamp_regressions': summary['time']['input_order_regressions'],
    'pattern_cases': summary['pattern_ground_truth'].get('case_count'),
    'pattern_typologies': summary['pattern_ground_truth'].get('typology_count'),
})

In [ ]:
quality = pd.read_csv(REPORT / 'quality_checks.csv')
columns = pd.read_csv(REPORT / 'column_profile.csv')
display(quality)
display(columns.sort_values('null_rate', ascending=False).head(15))

In [ ]:
for chart in ('class_balance.png', 'daily_volume.png', 'amount_distribution.png'):
    path = REPORT / chart
    if path.exists():
        display(Image(filename=str(path)))

## Case and typology coverage

Typology is privileged synthetic ground truth. Use it for held-out stratification and case metrics only—never as a predictor, score input, or reviewer-visible hint.

In [ ]:
typologies = pd.read_csv(REPORT / 'typology_profile.csv')
typologies.sort_values(['cases', 'transactions'], ascending=False)

## Leakage review

The dictionary combines measured types/null rates with the canonical feature contract. Empty null rates mean an expected field was absent from the profiled input, not that its null rate is zero.

In [ ]:
dictionary = pd.read_csv(DICTIONARY)
display(dictionary.head(12))
dictionary[dictionary['leakage_risk_flag'].astype(str).str.lower().eq('true')][
    ['feature_name', 'model_role', 'transformation_applied', 'description']
]

## Handoff to later sprints

Sprint 1 establishes data and evaluation contracts. It does not show that explanations help. Later experiments must compare every arm in `config/study.json` at identical queue capacity, freeze the reliability gate before test evaluation, include an ungated case-context ablation, and report null or harmful outcomes as well as positive ones.